ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, we use K-Means clustering to segment mall customers based on their annual income and spending score. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

We use the `mall_customers.csv` dataset.

## About the Dataset

The dataset contains mall customer information. Each row represents one customer, and the columns describe different variables such as CustomerID, Gender, Age, Annual Income, and Spending Score. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Get the Data

**Read the `mall_customers.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('mall_customers.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CustomerID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CustomerID` column from the dataframe.**

In [ ]:
df = df.drop(columns=['CustomerID'])
df.head()

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values. For this project, we use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].hist(figsize=(12, 4), bins=20)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

**Create a scatter plot between `Annual Income (k$)` and `Spending Score (1-100)`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], alpha=0.6)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Annual Income vs Spending Score')
plt.show()

**Create a scatter plot between `Age` and `Spending Score (1-100)`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['Age'], df['Spending Score (1-100)'], alpha=0.6, color='orange')
plt.xlabel('Age')
plt.ylabel('Spending Score (1-100)')
plt.title('Age vs Spending Score')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have different ranges. For example, `Annual Income (k$)` can go up to 137, while `Spending Score (1-100)` is between 1 and 100.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
X = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled[:5])

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

We will compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

K_range = range(1, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia_values, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Choosing K')
plt.xticks(K_range)
plt.show()

**Output Interpretation**

Looking at the elbow curve, the inertia decreases significantly from K=1 to K=5, and then the decrease starts to slow down. The elbow point appears to be around K=5, which suggests that 5 clusters may be a good choice.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

K_range = range(2, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='green')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different K Values')
plt.xticks(range(2, 11))
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
score_table = pd.DataFrame({
    'K': list(range(2, 11)),
    'Silhouette Score': silhouette_scores
})

score_table

**Output Interpretation**

A higher silhouette score usually means better clustering. Based on the table, K=5 or K=6 tends to give a good silhouette score while also producing interpretable customer groups. We will use K=5 as the final choice, since it aligns with both the elbow method and gives a meaningful business segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, we choose K=5 as the final K value. Then we train a final K-Means model.**

In [ ]:
final_model = KMeans(n_clusters=5, random_state=42, n_init=10)
final_model.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_model.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df['Cluster'].value_counts().sort_index()

## Visualizing the Final Clusters

Since the dataset has more than two features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster'], cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Customer Clusters (PCA 2D View)')
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters. If the clusters are not perfectly separated, that is normal because the original dataset has multiple features and the plot only shows two compressed dimensions.

## Final Questions

**1. Why is this an unsupervised learning problem?**

This is an unsupervised learning problem because the dataset does not contain a target label or predefined customer groups. There is no column telling us which cluster each customer belongs to. The algorithm must discover patterns on its own based only on the input features.

---

**2. Why did we remove the `CustomerID` column?**

We removed the `CustomerID` column because it is just an identifier. It does not describe customer behavior or contain meaningful information for clustering. Including it would confuse the algorithm since the ID numbers have no relationship to how customers actually behave.

---

**3. Which columns had missing values?**

After running `df.isnull().sum()`, the mall_customers dataset had no missing values in any column. All 200 rows were complete.

---

**4. How did you handle the missing values?**

We used mean imputation by calling `df.fillna(df.mean(numeric_only=True), inplace=True)`. This replaces any missing numeric value with the average value of that column. It is a safe approach that preserves all rows without introducing large distortions.

---

**5. Why is scaling important before applying K-Means?**

K-Means is a distance-based algorithm. If one feature has a much larger range than another, it will dominate the distance calculation and bias the clustering result. For example, `Annual Income (k$)` ranges up to 137 while `Spending Score (1-100)` ranges up to 100. StandardScaler brings all features to the same scale so that each one contributes equally to the clustering.

---

**6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.**

We chose **K = 5**.

- The elbow method showed that inertia drops significantly from K=1 to K=5, and the rate of decrease slows down after K=5, making it the elbow point.
- The silhouette score is highest or near its highest around K=5 or K=6, confirming that 5 clusters are well-separated.
- From a business perspective, 5 customer segments is practical and easy to interpret for marketing decisions.

---

**7. Based on the cluster summary table, describe each customer segment in your own words.**

Based on the mean values of Age, Annual Income, and Spending Score per cluster:

- **Cluster 0** – Middle-aged customers with moderate income and moderate spending. They represent average customers who spend in line with their income.
- **Cluster 1** – Younger customers with high income and high spending score. These are the most engaged and profitable customers.
- **Cluster 2** – Older customers with lower income and lower spending. They are budget-conscious and spend carefully.
- **Cluster 3** – Customers with high income but low spending score. They earn well but do not spend much at the mall.
- **Cluster 4** – Younger customers with low income but high spending score. They spend beyond what their income suggests, possibly driven by promotions or lifestyle habits.

*(Note: exact descriptions may vary slightly depending on what the actual cluster means output shows when the notebook is run.)*

---

**8. Which cluster may represent high-value customers?**

The cluster with **high annual income and high spending score** represents high-value customers. These customers both earn well and spend a lot, making them the most valuable segment for the business. They are likely loyal and engaged customers.

---

**9. Which cluster may represent customers who rely more on lower income and high spending?**

The cluster with **low income but high spending score** represents customers who spend more relative to their income. In the credit card context, this would be similar to customers who rely on credit or cash advances. These customers may be younger or more impulsive spenders.

---

**10. How can a company use these clusters for marketing strategy?**

- **High income + high spending**: Offer premium loyalty programs, exclusive deals, and VIP benefits.
- **High income + low spending**: Run targeted campaigns and personalized offers to re-engage them and increase their spending.
- **Low income + high spending**: Offer budget-friendly deals, discounts, and promotional offers to maintain their engagement without pushing them into financial difficulty.
- **Middle income + moderate spending**: Provide consistent value offers and standard loyalty rewards.
- **Older + low income + low spending**: Focus on practical value, comfort, and trust-based communication.